# 23 — Statistical rigour of the background: epoch dependence, local background, tail extrapolation, rate limits

**Author:** Giacomo Moretti · *Statistics and Data Analysis*, Università di Milano-Bicocca, A.Y. 2025–2026

**What it does.** Extensions 3, 4, 7 and 52 on the sine-Gaussian bank statistic of notebook 03 (all eight chunks, windows scanned and cached):

* **3. Epoch dependence.** Leave-one-chunk-out calibration: the threshold of FAP 10⁻² / 10⁻³ is set on the seven other chunks and the observed FAP is measured in the held-out chunk (binomial/Clopper–Pearson intervals); relation with the share of glitchy windows and with the PSD level of the chunk.
* **4. Time-local background.** The threshold of every window comes from its ±N neighbouring windows of the same chunk (minutes), instead of the global background; calibration across chunks and efficiency of injected signals with the local threshold.
* **7. Tail extrapolation.** A generalised Pareto distribution is fitted to the exceedances of the pooled train+validation background above its 95 % quantile and used to extrapolate the FAP below the empirical resolution, checked against the held-out test chunks and compared with the Gaussian formula.
* **52. Rate limits.** Poisson upper limits on the rate of detectable signals (Feldman–Cousins, with background) from the search of notebook 14 and from the zero-lag count, as a function of the signal SNR.

**What it does not do.** Single detector H1 (except the rate limit, which uses the H1+L1 search of notebook 14); the "local" background still contains the signal windows themselves in real data; the rate limits are for signals of a given optimal SNR, not an astrophysical rate density.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import gwosc_tools as g
g.style()
CH = {i: g.load_chunk(i) for i in range(len(g.chunks()))}
def cscan(n): d = np.load(os.path.join(g.CACHE, f"scan03_{n}.npz")); return d["rho2"]
RHO, CID, POS = {}, {}, {}
for sp in ("train", "val", "test"):
    v, cid, _ = g.load_split(sp); r = cscan(sp)
    for c in np.unique(cid): RHO[c] = r[cid == c]
NCH = sorted(RHO); print({c: len(RHO[c]) for c in NCH})

{np.int64(0): 4056, np.int64(1): 4056, np.int64(2): 4056, np.int64(3): 4056, np.int64(4): 4056, np.int64(5): 4056, np.int64(6): 4056, np.int64(7): 4056}


## 3. Epoch dependence: leave-one-chunk-out

In [2]:
def cp(k, n, cl=0.6827):
    a = (1 - cl) / 2; return (stats.beta.ppf(a, k, n - k + 1) if k > 0 else 0.0, stats.beta.ppf(1 - a, k + 1, n - k))
q999 = np.quantile(np.concatenate([RHO[c] for c in NCH]), 0.999)
rows = []
for c in NCH:
    other = np.concatenate([RHO[k] for k in NCH if k != c])
    for far in (1e-2, 1e-3):
        thr = g.threshold_at_far(other, far); k = int((RHO[c] > thr).sum()); n = len(RHO[c]); lo, hi = cp(k, n)
        rows.append(dict(chunk=c, FAP=far, observed=k / n, lo68=lo, hi68=hi, ratio=k / n / far, glitch_share=float((RHO[c] > q999).mean()), psd_rel=json.load(open(os.path.join(g.RESULTS, "01_summary.json")))["rel_sensitivity"][c]))
LOO = pd.DataFrame(rows); LOO.pivot_table(index="chunk", columns="FAP", values="ratio").round(2).join(LOO[LOO.FAP == 1e-2].set_index("chunk")[["glitch_share", "psd_rel"]].round(4))

,0.001,0.01,glitch_share,psd_rel
chunk,,,,
0,0.49,0.59,0.0005,1.0000
1,0.49,0.44,0.0005,1.0001
2,4.93,4.29,0.0020,1.0430
3,0.99,0.30,0.0010,1.0591
4,1.23,0.74,0.0012,1.0546
5,0.49,0.86,0.0005,1.0739
6,0.99,0.64,0.0010,1.0601
7,1.48,0.52,0.0015,1.0535


In [3]:
print("Spearman correlation across the 8 chunks of the FAP ratio at 1e-2 with the glitch share: %.2f, with the PSD level: %.2f" % (stats.spearmanr(LOO[LOO.FAP == 1e-2].ratio, LOO[LOO.FAP == 1e-2].glitch_share)[0], stats.spearmanr(LOO[LOO.FAP == 1e-2].ratio, LOO[LOO.FAP == 1e-2].psd_rel)[0]))
print("ratio observed/nominal: min %.2f max %.2f at 1e-2;  min %.2f max %.2f at 1e-3" % tuple(np.r_[LOO[LOO.FAP == 1e-2].ratio.agg(["min", "max"]), LOO[LOO.FAP == 1e-3].ratio.agg(["min", "max"])]))

Spearman correlation across the 8 chunks of the FAP ratio at 1e-2 with the glitch share: 0.29, with the PSD level: 0.21
ratio observed/nominal: min 0.30 max 4.29 at 1e-2;  min 0.49 max 4.93 at 1e-3


## 4. Time-local background (±N windows, same chunk)

In [4]:
def local_threshold(x, far, half, gap=5):
    # threshold of every window from the windows within +-half, excluding +-gap windows around it (a local signal must not raise its own threshold)
    n = len(x); thr = np.empty(n); order = {}
    for i in range(n):
        a, b = max(0, i - half), min(n, i + half + 1); idx = np.r_[np.arange(a, max(a, i - gap)), np.arange(min(b, i + gap + 1), b)]
        thr[i] = np.quantile(x[idx], 1 - far)
    return thr
rows = []
for half in (150, 300, 600):
    for far in (1e-2,):
        for c in NCH:
            thr = local_threshold(RHO[c], far, half); rows.append(dict(half_window_s=half, chunk=c, observed=float((RHO[c] > thr).mean())))
LOC = pd.DataFrame(rows); LP = LOC.pivot_table(index="chunk", columns="half_window_s", values="observed")
glob = {c: float((RHO[c] > g.threshold_at_far(np.concatenate([RHO[k] for k in NCH if k != c]), 1e-2)).mean()) for c in NCH}
LP["leave-one-out global"] = pd.Series(glob); print("observed FAP at nominal 1e-2 (per-chunk spread = std across chunks):"); display(LP.round(4)); print(LP.std().round(4))

observed FAP at nominal 1e-2 (per-chunk spread = std across chunks):


half_window_s,150,300,600,leave-one-out global
chunk,,,,
0,0.0126,0.0116,0.0113,0.0059
1,0.0133,0.0126,0.0101,0.0044
2,0.0150,0.0133,0.0094,0.0429
3,0.0143,0.0121,0.0108,0.0030
4,0.0136,0.0113,0.0108,0.0074
5,0.0158,0.0133,0.0101,0.0086
6,0.0150,0.0123,0.0101,0.0064
7,0.0128,0.0108,0.0108,0.0052


half_window_s
150                     0.0012
300                     0.0009
600                     0.0006
leave-one-out global    0.0132
dtype: float64


In [5]:
# efficiency of injected sine-Gaussians with the local threshold (host window known), vs the global threshold from the other chunks
rng = np.random.default_rng(23); SNR = [3, 4, 5, 6, 7, 8, 10, 12]; res = {"local": {s: [] for s in SNR}, "global": {s: [] for s in SNR}}
B = {c: g.SGBank(CH[c]["S1"]) for c in NCH}
for c in NCH:
    v = CH[c]["v"]; thr_l = local_threshold(RHO[c], 1e-2, 300); thr_g = g.threshold_at_far(np.concatenate([RHO[k] for k in NCH if k != c]), 1e-2)
    for s in SNR:
        i = rng.choice(len(v), 150, replace=False); ser, _, _ = g.inject(v[i], "sg", float(s), rng, CH[c]["S1"]); r = B[c].scan(g.to_bins(ser))["rho2"]
        res["local"][s] += list(r > thr_l[i]); res["global"][s] += list(r > thr_g)
EF = pd.DataFrame({k: {s: float(np.mean(v)) for s, v in d.items()} for k, d in res.items()}); print("efficiency of sg injections at nominal FAP 1e-2 (all chunks):"); EF.round(3)

efficiency of sg injections at nominal FAP 1e-2 (all chunks):


,local,global
3,0.035,0.023
4,0.178,0.129
5,0.482,0.442
6,0.777,0.776
7,0.929,0.959
8,0.976,0.997
10,0.989,1.000
12,1.000,1.000


## 7. Tail extrapolation with a generalised Pareto distribution

In [6]:
pool = np.concatenate([RHO[c] for c in g.SPLIT["train"] + g.SPLIT["val"]]); test = np.concatenate([RHO[c] for c in g.SPLIT["test"]])
u = np.quantile(pool, 0.95); exc = pool[pool > u] - u; shape, loc, scale = stats.genpareto.fit(exc, floc=0)
def gpd_far(x, u=u, shape=shape, scale=scale, p_u=0.05): return p_u * stats.genpareto.sf(x - u, shape, scale=scale)
rows = []
for far in (1e-2, 1e-3, 3e-4, 1e-4, 1e-5, 1e-6):
    thr_emp = g.threshold_at_far(pool, far) if far >= 1.5 / len(pool) else np.nan
    thr_gpd = u + stats.genpareto.isf(far / 0.05, shape, scale=scale)
    rows.append(dict(FAP=far, thr_empirical=thr_emp, thr_GPD=thr_gpd, observed_on_test_at_GPD_thr=float((test > thr_gpd).mean()), test_expected=far))
GP = pd.DataFrame(rows); print("GPD shape xi = %.3f, scale = %.2f, u = %.1f, %d exceedances" % (shape, scale, u, len(exc)))
# bootstrap of the extrapolated threshold at 1e-5
rb = np.random.default_rng(1); th = []
for _ in range(300):
    pb = rb.choice(pool, len(pool)); ub = np.quantile(pb, .95); eb = pb[pb > ub] - ub; sh, _, sc = stats.genpareto.fit(eb, floc=0); th.append(ub + stats.genpareto.isf(1e-5 / 0.05, sh, scale=sc))
print("threshold at FAP 1e-5: %.1f  (68%% bootstrap: %.1f - %.1f);  Gaussian-noise formula with N_eff = 6800: %.1f" % (u + stats.genpareto.isf(1e-5 / 0.05, shape, scale=scale), *np.percentile(th, [16, 84]), -2 * np.log(1 - (1 - 1e-5) ** (1 / 6800))))
GP.round(4)

GPD shape xi = 0.905, scale = 2.33, u = 24.4, 1217 exceedances


threshold at FAP 1e-5: 5743.6  (68% bootstrap: 3498.4 - 8750.4);  Gaussian-noise formula with N_eff = 6800: 40.7


,FAP,thr_empirical,thr_GPD,observed_on_test_at_GPD_thr,test_expected
0,0.0100,31.3455,32.8477,0.0023,0.0100
1,0.0010,102.0186,110.5497,0.0012,0.0010
2,0.0003,414.4896,285.5670,0.0009,0.0003
3,0.0001,17537.5735,734.4299,0.0009,0.0001
4,0.0000,NaN,5743.6486,0.0001,0.0000
5,0.0000,NaN,45963.3484,0.0000,0.0000


## 52. Poisson upper limits on the rate of detectable signals

In [7]:
def fc_upper(n_obs, b, cl=0.90):
    # Feldman-Cousins upper limit on the signal mean for n_obs events and a known background b
    mus = np.arange(0, 80, 0.02); acc = []
    for mu in mus:
        nmax = int(mu + b + 8 * np.sqrt(mu + b) + 10); n = np.arange(0, nmax); p = stats.poisson.pmf(n, mu + b)
        mu_best = np.maximum(0, n - b); pb = stats.poisson.pmf(n, mu_best + b); R = p / pb; o = np.argsort(-R); cum = np.cumsum(p[o]); k = np.searchsorted(cum, cl) + 1
        acc.append(n_obs in set(n[o][:k]))
    acc = np.array(acc); return mus[acc].max() if acc.any() else np.nan
s14 = g.load_json("14_summary.json"); T_h = s14["searched_hours"]
print("zero-lag: %d triggers above S=%g, expected from slides %.1f; searched time %.2f h" % (s14["n_zero_lag"], s14["floor"], s14["n_bkg_expected"], T_h))
UL = {"zero events, no background": 2.303, "FC 90% (n=62, b=63.1)": fc_upper(62, 63.1)}
eff = {r["rho_net"]: r["detected_FAR<1e3/yr"] for r in s14["recovery"]}
rows = [dict(case=k, mu90=v, **{f"rate90 per h at rho_net={r} (eff {e:.1f})": v / (T_h * e) for r, e in eff.items() if e > 0}) for k, v in UL.items()]
RL = pd.DataFrame(rows); print("90 % CL upper limit on the number of detectable signals in the searched data, and on the rate [per hour] for signals of a given network SNR (efficiency from notebook 14):"); RL.round(2).T

zero-lag: 62 triggers above S=6, expected from slides 63.1; searched time 2.25 h


90 % CL upper limit on the number of detectable signals in the searched data, and on the rate [per hour] for signals of a given network SNR (efficiency from notebook 14):


,0,1
case,"zero events, no background","FC 90% (n=62, b=63.1)"
mu90,2.3,12.96
rate90 per h at rho_net=8 (eff 0.1),10.22,57.52
rate90 per h at rho_net=10 (eff 0.4),2.56,14.38
rate90 per h at rho_net=12 (eff 0.7),1.46,8.22
rate90 per h at rho_net=15 (eff 1.0),1.02,5.75


In [8]:
g.save_json(dict(loo=LOO.to_dict("records"), local_background=LP.reset_index().to_dict("records"), efficiency_local_vs_global=EF.reset_index().to_dict("records"),
                 gpd=dict(u=float(u), shape=float(shape), scale=float(scale), table=GP.to_dict("records"), thr_1e5_bootstrap=[float(x) for x in np.percentile(th, [16, 50, 84])]),
                 rate_limits=RL.to_dict("records")), "23_summary.json"); print("saved")

saved


## Output and next step

`results/23_summary.json`. Next: rare-event Monte Carlo for the background tail (extension 51) and the seed study of the networks (extension 2).